# Introduction

In this notebook, we will explore the OpenAI library. We will also acquire data in different ways to inform our queries.

The setup is not different from what we have done before.

In [10]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [8]:
import sys
sys.path.append('../../05_src/')

In [9]:
import os
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

2. We create an API call and store the result in the variable `client`. Notice that the call specifies the model that we want to use, as well as an input. This is a simple call; the [responses API can handle more complex calls](https://platform.openai.com/docs/api-reference/responses).

# Longer Context

Thus far, we have created our context and prompt manually. We can use any of Python's text manipulation tools to create documents. 

For instance, in the code below, we will use the requests library to download a book from Project Gutenberg and include it in the context. Some useful links are:

- Python's [request](https://pypi.org/project/requests/) library.
- Python's request library [documentation](https://requests.readthedocs.io/en/latest/).

In [3]:
import requests
file_url = 'https://www.gutenberg.org/cache/epub/223/pg223.txt'
book = requests.get(file_url)

In [ ]:
book

The code below shows how to retrieve the response headers. 

In [11]:
dict(book.headers)

{'date': 'Sun, 28 Jun 2026 12:57:09 GMT',
 'server': 'Apache',
 'last-modified': 'Mon, 01 Jun 2026 08:40:55 GMT',
 'accept-ranges': 'bytes',
 'content-length': '434825',
 'x-backend': 'gutenweb5',
 'content-type': 'text/plain; charset=utf-8'}

We can also obtain the payload using `book.text` or `book.content`.

In [13]:
print(book.text[:100])

The Project Gutenberg eBook of The wisdom of Father Brown
    
This eBook is for the use of anyone


With added content, we can also write more interesting prompts. Notice the use of Python's f-strings or formatted strings. 

When using formatted strings, remember two things:

+ Formatted strings are prefixed with f:  `f'This is a formatted string: {some_variable}.`
+ You can enclose variable names in curly brackes, `{...}`, and their values will appear in formatted strings: `f'The variable value is {variable}.`

In [20]:
prompt = f"""
    You are a high school literature teacher. 
    Given the following context from a book, do the following:
    
    1. Identify the book's title and author.
    2. Determine how many stories are included in the book.
    3. Summarize concisely in no more than 5 paragraphs the first story of the book called "The Absence of Mr Glass".
        
    The book is the following: 
    <book>
    </book>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Number of Stories: <number_of_stories>
    Summary: <summary>
"""

In [21]:
from IPython.display import display, Markdown
display(Markdown(f"### Prompt:\n{prompt}"))


### Prompt:

    You are a high school literature teacher. 
    Given the following context from a book, do the following:

    1. Identify the book's title and author.
    2. Determine how many stories are included in the book.
    3. Summarize concisely in no more than 5 paragraphs the first story of the book called "The Absence of Mr Glass".

    The book is the following: 
    <book>
    </book>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Number of Stories: <number_of_stories>
    Summary: <summary>


In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not set. Add it to ../../05_src/.env or export OPENAI_API_KEY in your environment.")

try:
    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )
except Exception as e:
    if "Incorrect API key provided" in str(e) or "401" in str(e):
        raise RuntimeError("Authentication failed: OPENAI_API_KEY is incorrect. Update your key in ../../05_src/.secrets or export a valid OPENAI_API_KEY.") from e
    raise


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: <if avia**********************************nAI>. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

We use `Ipython` to format the output using markdown and create a friendlier display.

In [ ]:
display(Markdown(response.output_text))

# Adding a Developer Prompt

Using the OpenAI library, we can specify prompts for different roles:

+ *Developer* messages are instructions provided by the application developer, prioritized ahead of user messages. In previous iterations of the API, these were called *System* messages.
+ *User* messages are instructions provided by an end user, prioritized below developer messages.
+ *Assistant* messages are generated by the model.

In [ ]:
system_prompt = "You are a specialist in ancient fables who speaks like Bad Bunny."

In [ ]:
the_ant = """
Ants were once men and made their living by tilling the soil. But, not content with the results of their own work, they were always casting longing eyes upon the crops and fruits of their neighbours, which they stole, whenever they got the chance, and added to their own store. At last their covetousness made Jupiter so angry that he changed them into Ants. But, though their forms were changed, their nature remained the same: and so, to this day, they go about among the cornfields and gather the fruits of others’ labour, and store them up for their own use.
"""

prompt = f"""
    Please, extract the main message of this fable written by Aesop.
    The fable is the following: 
    <fable>
    {the_ant}
    </fable>
"""

In [ ]:
response = client.responses.create(
    model = MODEL, 
    instructions = system_prompt,
    input = prompt,
)

In [ ]:
display(Markdown(response.output_text))